This script retrieves the most up-to-date copies of the sharepoint lists from AWS and checks for incosistencies/missing information

In [ ]:
import logging
import os
import boto3
import getpass
import pandas as pd
import numpy as np


from typing import List, Dict, Set, Union, Tuple, Iterator, Optional
from botocore.exceptions import ClientError
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass, field

# Configure logging with a more detailed format
logging.basicConfig(
level=logging.INFO,
format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class AWSCredentials:
    access_key_id: str
    secret_access_key: str

@classmethod
def from_user_input(cls) -> 'AWSCredentials':
    """Securely prompt user for AWS credentials."""
    access_key = getpass.getpass("Enter AWS Access Key ID: ")
    secret_key = getpass.getpass("Enter AWS Secret Access Key: ")
    return cls(access_key, secret_key)

class S3Client:
    def __init__(self, credentials: Optional[AWSCredentials] = None):
        self.client = self._initialize_client(credentials)

    def _initialize_client(self, credentials: Optional[AWSCredentials]) -> boto3.client:
        """Initialize S3 client with credentials from env vars, provided credentials, or user input."""
        if credentials is None:
            # Try environment variables first
            access_key = os.getenv("AWS_ACCESS_KEY_ID")
            secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
            
            if not access_key or not secret_key:
                logger.info("AWS credentials not found in environment variables. Please enter them manually.")
                credentials = AWSCredentials.from_user_input()
            else:
                credentials = AWSCredentials(access_key, secret_key)

        try:
            client = boto3.client(
                "s3",
                aws_access_key_id=credentials.access_key_id,
                aws_secret_access_key=credentials.secret_access_key,
            )
            # Test the credentials by making a simple API call
            client.list_buckets()
            logger.info("Successfully authenticated with AWS")
            return client
        except ClientError as e:
            logger.error("Failed to authenticate with AWS")
            if "InvalidAccessKeyId" in str(e) or "SignatureDoesNotMatch" in str(e):
                logger.error("Invalid credentials provided. Please try again.")
                credentials = AWSCredentials.from_user_input()
                return self._initialize_client(credentials)
            raise

    def list_objects(self, bucket: str, prefix: str = "", suffix: str = "") -> Iterator[dict]:
        """List objects in an S3 bucket with optional prefix and suffix filtering."""
        paginator = self.client.get_paginator("list_objects_v2")
        
        for prefix_item in [prefix] if isinstance(prefix, str) else prefix:
            try:
                for page in paginator.paginate(Bucket=bucket, Prefix=prefix_item):
                    if "Contents" not in page:
                        continue
                    
                    for obj in page["Contents"]:
                        if obj["Key"].endswith(suffix):
                            yield obj
            except ClientError as e:
                logger.error(f"Error listing objects: {e}")
                raise

    def download_file(self, bucket: str, key: str, filename: Path, version_id: Optional[str] = None) -> None:
        """Download a file from S3 with progress tracking."""
        try:
            kwargs = {"Bucket": bucket, "Key": key}
            if version_id:
                kwargs["VersionId"] = version_id

            object_size = self.client.head_object(**kwargs)["ContentLength"]
            
            with tqdm(total=object_size, unit='B', unit_scale=True, desc=str(filename)) as pbar:
                self.client.download_file(
                    Bucket=bucket,
                    Key=key,
                    Filename=str(filename),
                    Callback=pbar.update
                )
        except ClientError as e:
            logger.error(f"Error downloading {key}: {e}")
            raise

    def get_movies_self(self, prefix: str = "") -> pd.DataFrame:
        """Get DataFrame of movie files in S3 bucket with their sizes.    
        Args:
            prefix: Optional prefix to filter S3 objects            
        Returns:
            DataFrame with columns 'Key' and 'Size' (in bytes)
        """
        # Get all objects matching the prefix and movie extensions
        objects = self.s3_client.list_objects(
            self.bucket,
            prefix=prefix,
            suffix=tuple(self.MOVIE_EXTENSIONS)
        )
        
        # Extract both keys and sizes
        movie_data = [
            {
                'Key': obj['Key'],
                'Size': obj['Size']  # Size in bytes
            }
            for obj in objects
        ]
        
        return pd.DataFrame(movie_data)

    def _download_videos(self, keys: pd.Series) -> List[Path]:
        """Download all videos for a drop."""
        downloaded_files = []
        for key in keys:
            local_path = self.download_dir / Path(key).name
            self.s3_client.download_file(self.bucket, key, local_path)
            downloaded_files.append(local_path)
        return downloaded_files

@dataclass
class CSVCollection:
    """Container for multiple DataFrames loaded from CSV files."""
    dataframes: Dict[str, pd.DataFrame] = field(default_factory=dict)
    
    def get_df(self, key: str) -> pd.DataFrame:
        if key not in self.dataframes:
            raise KeyError(f"No DataFrame found for key: {key}")
        return self.dataframes[key]
    
    def list_dataframes(self) -> List[str]:
        """List all available DataFrame keys."""
        return list(self.dataframes.keys())

class CSVValidator:
    def __init__(self, s3_client: S3Client):
        self.s3_client = s3_client
        self.logger = logging.getLogger(__name__)
        self.csv_collection = None

    def load_csvs_from_prefix(self) -> CSVCollection:
        """
        Load all CSV files from a given S3 prefix into a dictionary of DataFrames.        
                    
        Returns:
            CSVCollection containing DataFrames mapped by their filenames
        """
        try:
            # Try environment variables first
            bucket = os.getenv("S3_BUCKET")
            
            if not bucket:
                logger.info("AWS Bucket not found in environment variables. Please enter it manually.")
                bucket = getpass.getpass("Enter AWS Bucket name:")
                
            # Try environment variables first
            prefix = os.getenv("S3_SHAREPOINT_CSVS")
            
            if not prefix:
                logger.info("AWS Key with csv files copies of the sharepoint lists not found in environment variables. Please enter it manually.")
                prefix = getpass.getpass("AWS Key of with csv files:")
            
            # List all CSV objects with the given prefix
            csv_objects = self.s3_client.list_objects(bucket, prefix=prefix, suffix=".csv")
            
            # Create a new CSV collection
            self.csv_collection = CSVCollection()
            
            for obj in csv_objects:
                key = obj["Key"]
                filename = Path(key).name
                
                # Create a temporary file to store the CSV
                temp_path = Path(f"temp_{filename}")
                
                try:
                    # Download and load the CSV
                    self.s3_client.download_file(bucket, key, temp_path)
                    df = pd.read_csv(temp_path)
                    
                    # Store in the collection using the filename (without .csv) as the key
                    df_key = filename.replace('.csv', '')
                    self.csv_collection.dataframes[df_key] = df
                    
                    self.logger.info(f"Successfully loaded {filename}")
                    
                finally:
                    # Clean up temporary file
                    if temp_path.exists():
                        temp_path.unlink()
            
            return self.csv_collection
            
        except Exception as e:
            self.logger.error(f"Error loading CSVs from prefix {prefix}: {e}")
            raise

    def check_missing_values(df: pd.DataFrame) -> Dict[str, int]:
        """
        Check for missing values in columns.
        
        Args:
            df: Input DataFrame to check for missing values
            
        Returns:
            Dictionary mapping column names to count of missing values
        """
        return {col: df[col].isna().sum() for col in df.columns}

    def check_unique_values(df: pd.DataFrame, unique_columns: List[Union[str, List[str]]]) -> Dict[str, List]:
        """
        Check for duplicate values in columns that should be unique.
        Supports both single columns and combinations of columns.
        
        Args:
            df: Input DataFrame to check for duplicates
            unique_columns: List of either column names or lists of column names that should contain unique values
            
        Returns:
            Dictionary mapping column names (or composite column names) to lists of duplicate values
        """
        duplicates = {}
        
        for col_spec in unique_columns:
            if isinstance(col_spec, str):
                # Single column check
                if col_spec in df.columns:
                    dups = df[df[col_spec].duplicated()][col_spec].tolist()
                    if dups:
                        duplicates[col_spec] = dups
            elif isinstance(col_spec, list) and len(col_spec) > 1:
                # Composite columns check
                if all(col in df.columns for col in col_spec):
                    # Find rows with duplicate combinations
                    dup_mask = df.duplicated(subset=col_spec, keep=False)
                    if dup_mask.any():
                        # Create a compound key of the duplicated values
                        dup_rows = df[dup_mask][col_spec]
                        dup_combinations = [
                            tuple(row) for row in dup_rows.values.tolist()
                        ]
                        # Use '-'.join for the dictionary key
                        key_name = '-'.join(col_spec)
                        duplicates[key_name] = dup_combinations
        
        return duplicates

    def check_reference_integrity(
        source_df: pd.DataFrame,
        reference_df: pd.DataFrame,
        source_column: str,
        reference_column: str
    ) -> List:
        """
        Check if all values in source column exist in reference column.
        
        Args:
            source_df: DataFrame containing the source values
            reference_df: DataFrame containing the reference values
            source_column: Column name in source_df to check
            reference_column: Column name in reference_df to check against
            
        Returns:
            List of values from source_column that don't exist in reference_column
            
        Raises:
            ValueError: If specified columns are not found in the DataFrames
        """
        if source_column not in source_df.columns:
            raise ValueError(f"Source column '{source_column}' not found")
        if reference_column not in reference_df.columns:
            raise ValueError(f"Reference column '{reference_column}' not found")
                
        source_values = set(source_df[source_column].dropna())
        reference_values = set(reference_df[reference_column].dropna())
        
        return list(source_values - reference_values)

    class DataValidator:
        def __init__(self, csv_collection):
            self.csv_collection = csv_collection

        def validate_loaded_csvs(
            self,
            unique_columns: Dict[str, List[Union[str, List[str]]]] = {
                'BUV Deployment': ['DropID', 'fileName', 'LinkToVideoFile'],
                'BUV Survey Metadata': ['SurveyID', 'SurveyName'],
                'BUV Survey Sites': ['SiteID', ['Latitude', 'Longitude']],  # Composite unique constraint
                'Marine Reserves': ['Title', 'SurveyLocationAcronym']
            },
            reference_mappings: Dict[str, Dict[str, str]] = {
                'BUV Deployment': {
                    'reference': 'BUV Survey Metadata',
                    'base_on': 'DropID',
                    'reference_on': 'DropID'
                },
                'BUV Survey Metadata': {
                    'reference': 'BUV Survey Sites',
                    'base_on': 'SiteID',
                    'reference_on': 'SiteID'
                },
                'BUV Survey Sites': {
                    'reference': 'Marine Reserves',
                    'base_on': 'LinkToMarineReserve',
                    'reference_on': 'Title'
                }
            }
        ) -> Dict:
            """
            Validate loaded CSVs using the csv_collection.
            
            Args:
                unique_columns: Dictionary mapping CSV names to lists of either single columns 
                            or lists of columns that should have unique values
                reference_mappings: Dictionary defining reference relationships between DataFrames
                
            Returns:
                Dictionary containing validation results for missing values, duplicates, and reference integrity
                
            Raises:
                ValueError: If no CSVs are loaded
            """
            if not self.csv_collection:
                raise ValueError("No CSVs loaded. Call load_csvs_from_prefix first.")

            validation_results = {
                'missing_values': {},
                'duplicate_values': {},
                'reference_integrity': {}
            }

            for csv_name in self.csv_collection.keys():
                df = self.csv_collection.get_df(csv_name)

                # Check for missing values
                validation_results['missing_values'][csv_name] = check_missing_values(df)

                # Check for duplicate values if unique columns are specified for this CSV
                if csv_name in unique_columns:
                    validation_results['duplicate_values'][csv_name] = check_unique_values(
                        df, unique_columns[csv_name]
                    )

                # Check reference integrity if mappings are specified for this CSV
                if csv_name in reference_mappings:
                    mapping = reference_mappings[csv_name]
                    reference_df = self.csv_collection.get_df(mapping['reference'])
                    
                    validation_results['reference_integrity'][csv_name] = check_reference_integrity(
                        df,
                        reference_df,
                        mapping['base_on'],
                        mapping['reference_on']
                    )

            return validation_results

    def check_and_export_differences(self, df: pd.DataFrame, output_path: str = 'column_differences.csv') -> tuple[pd.DataFrame, bool]:
        """
        Identifies duplicate columns with different values and exports them to CSV.
        Returns merged DataFrame with identical columns combined.
        
        Parameters:
        df (pandas.DataFrame): Input DataFrame with potential duplicate columns
        output_path (str): Path where to save differences CSV
        
        Returns:
        tuple[pandas.DataFrame, bool]: (Cleaned DataFrame with identical columns merged, Whether differences were found)
        """
        # Create a copy to avoid modifying the original DataFrame
        df_cleaned = df.copy()
        has_differences = False
        diff_rows = []
        
        # Get base column names (without _x, _y suffixes)
        base_columns = set()
        for col in df.columns:  # Changed from self.columns to df.columns
            if col.endswith('_x') or col.endswith('_y'):
                base_columns.add(col[:-2])
        
        # Process each base column
        for base_col in base_columns:
            x_col = f"{base_col}_x"
            y_col = f"{base_col}_y"
            
            # Check if both columns exist
            if x_col in df.columns and y_col in df.columns:  # Changed from self.columns to df.columns
                # Convert columns to same type and compare
                try:
                    # Try to convert to numeric if possible
                    x_series = pd.to_numeric(df[x_col], errors='ignore')
                    y_series = pd.to_numeric(df[y_col], errors='ignore')
                    
                    # Compare values (handling NaN values properly)
                    are_equal = x_series.equals(y_series)
                    
                    if not are_equal:
                        # Double check with string comparison for non-numeric values
                        x_str = df[x_col].astype(str)
                        y_str = df[y_col].astype(str)
                        are_equal = x_str.equals(y_str)
                except Exception as e:
                    self.logger.warning(f"Error comparing {x_col} and {y_col}: {str(e)}")
                    are_equal = False
                
                if are_equal:
                    # Keep one column and rename it
                    df_cleaned[base_col] = df_cleaned[x_col]
                    df_cleaned = df_cleaned.drop([x_col, y_col], axis=1)
                else:
                    # temporary workaround to keep on testing 
                    df_cleaned[base_col] = df_cleaned[x_col]
                    df_cleaned = df_cleaned.drop([x_col, y_col], axis=1)
                    
                    # has_differences = True
                    # print(f"Warning: {x_col} and {y_col} have different values")
                    
                    # # Find rows where values differ
                    # mask = ~(x_series.eq(y_series))
                    # diff_self = self[mask].copy()
                    
                    # # Get ID columns
                    # id_columns = [col for col in self.columns if 'ID' in col.upper() 
                    #             and not col.endswith('_x') 
                    #             and not col.endswith('_y')]
                    
                    # # Create rows for the difference report
                    # for idx in diff_self.index:
                    #     row_data = {
                    #         'Column': base_col,
                    #         'Value_x': str(diff_self.at[idx, x_col]),  # Convert to string to handle all types
                    #         'Value_y': str(diff_self.at[idx, y_col]),
                    #         'Row_Index': idx
                    #     }
                    #     for id_col in id_columns:
                    #         row_data[id_col] = diff_self.at[idx, id_col]
                    #     diff_rows.append(row_data)
        
        # Export differences if found
        if diff_rows:
            diff_df = pd.DataFrame(diff_rows)
            diff_df['Keep_Value'] = ''  # Column for manual input
            
            # Reorder columns to put IDs first
            id_columns = [col for col in diff_df.columns if 'ID' in col.upper()]
            other_columns = [col for col in diff_df.columns if col not in id_columns]
            diff_df = diff_df[id_columns + other_columns]
            
            diff_df.to_csv(output_path, index=False)
            print(f"Differences exported to {output_path}")
        
        return df_cleaned, has_differences

    def merge_buv_metadata(self) -> pd.DataFrame:
        """
        Performs multiple merges with difference checking after each merge.
        
        Returns:
        pandas.DataFrame: Final merged and cleaned DataFrame
        """
        if not self.csv_collection:
            raise ValueError("No CSVs loaded. Call load_csvs_from_prefix first.")

        has_any_differences = False
        
        # First merge
        df = self.csv_collection.get_df("BUV Deployment").merge(
            self.csv_collection.get_df("BUV Survey Metadata"), 
            on="SurveyID", 
            how="left"
        )
        df, has_diff = self.check_and_export_differences(df, 'differences_merge_Deployment_Survey.csv')
        has_any_differences |= has_diff
        
        # Second merge
        df = df.merge(
            self.csv_collection.get_df("BUV Survey Sites"), 
            on="SiteID", 
            how="left"
        )
        df, has_diff = self.check_and_export_differences(df, 'differences_merge_Deployment_Survey_Sites.csv')
        has_any_differences |= has_diff
        
        # Third merge
        df = df.merge(
            self.csv_collection.get_df("Marine Reserves"), 
            left_on="LinkToMarineReserve", 
            right_on="Title", 
            how="left"
        )
        df, has_diff = self.check_and_export_differences(df, 'differences_merge_Deployment_Survey_Sites_MReserves.csv')
        has_any_differences |= has_diff
        
        if not has_any_differences:
            self.logger.info("No differences found in any merge operations")
        
        return df

# Initialize S3 client and validator
s3_client = S3Client()
validator = CSVValidator(s3_client)

# Load all CSVs from a common prefix
csv_collection = validator.load_csvs_from_prefix()

# Print available DataFrames
print("Loaded CSV files:")
for key in csv_collection.list_dataframes():
    df = csv_collection.get_df(key)
    print(f"- {key}: {len(df)} rows")

# Create a DataFrame with all buv metadata
buv_metadata = validator.merge_buv_metadata()

In [ ]:
# Initialize S3 client and validator
s3_client = S3Client()
validator = CSVValidator(s3_client)

# Load all CSVs from a common prefix
csv_collection = validator.load_csvs_from_prefix()

# Print available DataFrames
print("Loaded CSV files:")
for key in csv_collection.list_dataframes():
    df = csv_collection.get_df(key)
    print(f"- {key}: {len(df)} rows")

# Create a DataFrame with all buv metadata
buv_metadata = validator.merge_buv_metadata() 

In [ ]:
buv_metadata